In [1]:
# Cell 1 — Setup
%pip install -q manuscript-ocr certifi natsort

import os
import glob

from natsort import natsorted

# macOS python.org / venv builds often ship without a usable CA bundle, so the
# library's HTTPS model-weight downloads fail with CERTIFICATE_VERIFY_FAILED.
# Point OpenSSL at certifi's bundle BEFORE any download is triggered below.
import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

from manuscript import Pipeline
from manuscript.detectors import EAST
from manuscript.recognizers import TRBA


# This notebook lives in notebooks/, but INPUT_DIR/OUTPUT_DIR are relative to the
# repo root. Walk up from the current working dir until we find the data folder,
# then chdir there so the notebook works regardless of where the kernel started.
def _find_repo_root(marker=os.path.join("data", "72.1.337")):
    d = os.getcwd()
    while True:
        if os.path.isdir(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError(
                f"Could not locate repo root containing {marker!r} starting from {os.getcwd()!r}"
            )
        d = parent


os.chdir(_find_repo_root())
print("Working directory:", os.getcwd())

INPUT_DIR = "data/72.1.337/"
OUTPUT_DIR = "data/ru/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Build the CoreML-accelerated pipeline ONCE (model load is expensive). Keeping
# it here means re-running the OCR loop cell does not reload the models.
detector = EAST(device="coreml")
recognizer = TRBA(device="coreml")
pipeline = Pipeline(detector=detector, recognizer=recognizer)
print("Pipeline ready (CoreML).")


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Working directory: /Users/okolobaxa/Documents/Projects/neo4gen
Pipeline ready (CoreML).


In [2]:
# Cell 2 — Discovery helpers (natural sort via natsort)
# Documents are sub-folders named "72.1.337-<N>"; each holds the scanned pages of
# one sub-document. natsorted() handles the numeric ordering for free:
#   - folders by N: 72.1.337-2 before 72.1.337-13 (NOT lexicographic);
#   - pages by folio number, then recto (f) before verso (r), since "f" < "r".


def sorted_pages(doc_dir):
    return natsorted(glob.glob(os.path.join(doc_dir, "*.jpg")))


def sorted_docs(input_dir):
    return natsorted(
        p for p in glob.glob(os.path.join(input_dir, "72.1.337-*")) if os.path.isdir(p)
    )


docs = sorted_docs(INPUT_DIR)
print(f"Found {len(docs)} sub-document(s):", [os.path.basename(d) for d in docs])
print("First document's pages:", [os.path.basename(p) for p in sorted_pages(docs[0])[:6]])

Found 10 sub-document(s): ['72.1.337-2', '72.1.337-8', '72.1.337-9', '72.1.337-13', '72.1.337-14', '72.1.337-39', '72.1.337-52', '72.1.337-53', '72.1.337-56', '72.1.337-94']
First document's pages: ['72.1.337.2f.jpg', '72.1.337.2r.jpg', '72.1.337.3f.jpg', '72.1.337.3r.jpg', '72.1.337.4f.jpg', '72.1.337.4r.jpg']


In [3]:
# Cell 3 — OCR every page, then combine each sub-document into a single .md
# --- Config ---
LIMIT_DOCS = None    # process only the first N sub-documents; set None to process all
REPROCESS = False    # True = re-OCR pages and rebuild the combined .md even if they exist

# Per-page OCR is cached here so a long run can resume page-by-page after an error.
# OUTPUT_DIR only ever receives the combined-per-document .md (no sub-folders there).
PAGES_CACHE_DIR = "data/ocr_pages/"
PAGE_SEP = "\n\n"    # how consecutive pages are joined inside the combined document

from IPython.display import display

docs = sorted_docs(INPUT_DIR)
if LIMIT_DOCS is not None:
    docs = docs[:LIMIT_DOCS]

ndocs = len(docs)
display(f"Processing {ndocs} sub-document(s)  (LIMIT_DOCS={LIMIT_DOCS}, REPROCESS={REPROCESS})")

for di, doc_dir in enumerate(docs, start=1):
    doc = os.path.basename(doc_dir)                       # e.g. 72.1.337-13
    combined_path = os.path.join(OUTPUT_DIR, f"{doc}.ru.md")

    if os.path.exists(combined_path) and not REPROCESS:
        display(f"[doc {di}/{ndocs}] skip  {doc} -> {doc}.ru.md (already combined)")
        continue

    pages = sorted_pages(doc_dir)
    cache_dir = os.path.join(PAGES_CACHE_DIR, doc)
    os.makedirs(cache_dir, exist_ok=True)

    page_texts = []
    for pi, page_path in enumerate(pages, start=1):
        page = os.path.basename(page_path)
        cache_path = os.path.join(cache_dir, page.replace(".jpg", ".ru.md"))

        if os.path.exists(cache_path) and not REPROCESS:
            display(f"[doc {di}/{ndocs}] [{pi}/{len(pages)}] cached {page}")
            with open(cache_path, encoding="utf-8") as f:
                page_texts.append(f.read())
            continue

        display(f"[doc {di}/{ndocs}] [{pi}/{len(pages)}] ocr    {page} ...")
        # No try/except by design: an OCR error halts the loop so the failing scan
        # can be inspected. Re-running resumes from the cached pages above.
        result = pipeline.predict(page_path)
        text = pipeline.get_text(result["page"])
        with open(cache_path, "w", encoding="utf-8") as f:
            f.write(text)
        page_texts.append(text)

    combined = PAGE_SEP.join(t.strip() for t in page_texts).strip() + "\n"
    with open(combined_path, "w", encoding="utf-8") as f:
        f.write(combined)
    display(f"[doc {di}/{ndocs}] done  {doc} -> {doc}.ru.md ({len(pages)} pages)")

display("Finished.")

'Processing 10 sub-document(s)  (LIMIT_DOCS=None, REPROCESS=False)'

'[doc 1/10] [1/22] ocr    72.1.337.2f.jpg ...'

[EAST] Device configuration:
  Requested device: coreml
  Requested providers: ['CoreMLExecutionProvider', 'CPUExecutionProvider']
  Active providers: ['CoreMLExecutionProvider', 'CPUExecutionProvider']
  Running on: CoreMLExecutionProvider


2026-06-19 12:32:47.686 Python[21443:3668974] 2026-06-19 12:32:47.686530 [W:onnxruntime:, coreml_execution_provider.cc:113 GetCapability] CoreMLExecutionProvider::GetCapability, number of partitions supported by CoreML: 4 number of nodes in the graph: 144 number of nodes supported by CoreML: 141
2026-06-19 12:32:50.145 Python[21443:3668974] 2026-06-19 12:32:50.145148 [W:onnxruntime:, coreml_execution_provider.cc:113 GetCapability] CoreMLExecutionProvider::GetCapability, number of partitions supported by CoreML: 172 number of nodes in the graph: 1347 number of nodes supported by CoreML: 1039


[TRBA] Device configuration:
  Requested device: coreml
  Requested providers: ['CoreMLExecutionProvider', 'CPUExecutionProvider']
  Active providers: ['CoreMLExecutionProvider', 'CPUExecutionProvider']
  Running on: CoreMLExecutionProvider


...

'[doc 10/10] [6/8] ocr    72.1.337.202r.jpg ...'

'[doc 10/10] [7/8] ocr    72.1.337.203f.jpg ...'

'[doc 10/10] [8/8] ocr    72.1.337.203r.jpg ...'

'[doc 10/10] done  72.1.337-94 -> 72.1.337-94.ru.md (8 pages)'

'Finished.'